In [1]:
!pip install -U -q transformers datasets bitsandbytes trl peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 110.0 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00:00:0100:01


In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-v0.3"
config_4bit = BitsAndBytesConfig(load_in_4bit=True)
model_4bit = AutoModelForCausalLM.from_pretrained(
                                        model_name,
                                        quantization_config=config_4bit,
                                        device_map="auto",
                                        trust_remote_code=True
                                                )

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left", trust_remote_code=True)                                                

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [4]:
from datasets import load_dataset
dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

README.md: 0.00B [00:00, ?B/s]

06102023.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2415 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>', '<PAD>', '<systems>', '<user>', '<assistant>']

In [9]:
tokens = ['<systems>', '<user>', '<assistant>']

for token in tokens:
    token_id = tokenizer.encode(token, add_special_tokens=False)
    print(token, token_id)

<systems> [1291, 7342, 29481, 29535]
<user> [1291, 2606, 29535]
<assistant> [1291, 1257, 11911, 29535]


In [10]:
tokenizer.pad_token

In [18]:
special_token = {
    'pad_token': '<PAD>',
    'additional_special_tokens': ['<systems>', '<user>', '<assistant>']
}

tokenizer.add_special_tokens(special_token)

0

In [20]:
tokens = ['<systems>', '<user>', '<assistant>', '<PAD>']

for token in tokens:
    token_id = tokenizer.encode(token, add_special_tokens=False)
    print(token, token_id)

<systems> [32769]
<user> [32770]
<assistant> [32771]
<PAD> [32768]


In [21]:
len(tokenizer)

32772

In [22]:
model_4bit.config.pad_token_id = tokenizer.pad_token_id

In [23]:
model_4bit.resize_token_embeddings(len(tokenizer))

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(32772, 4096)

In [24]:
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>', '<PAD>', '<systems>', '<user>', '<assistant>']

In [25]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [26]:
dataset['train'][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [27]:
new_template = """<s><system>{system_prompt}</s><user>{user_prompt}</s><assistant>{model_answer}</s>"""

def format_dataset(example):

    system_prompt = example['instruction']
    user_prompt = example['input']
    model_answer = example['output']

    formatted_text = new_template.format(
                    system_prompt = system_prompt,
                    user_prompt = user_prompt,
                    model_answer = model_answer,
    )

    return {'text': formatted_text}

format_dataset(dataset['train'][0])

{'text': "<s><system>translate this sentence in docker command</s><user>Give me a list of containers that have the Ubuntu image as their ancestor.</s><assistant>docker ps --filter 'ancestor=ubuntu'</s>"}

In [28]:
dataset = dataset.map(format_dataset)
dataset['train']['text'][0]

Map:   0%|          | 0/2415 [00:00<?, ? examples/s]

"<s><system>translate this sentence in docker command</s><user>Give me a list of containers that have the Ubuntu image as their ancestor.</s><assistant>docker ps --filter 'ancestor=ubuntu'</s>"